In [ ]:
from classification_models.keras import Classifiers
from collections import Counter
from copy import deepcopy
import gc
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import numpy as np
import os

import platform
import pandas as pd
import pytz
import sys

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, IsolationForest, GradientBoostingClassifier, AdaBoostClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score

from importlib.metadata import version # Version for reproducibility: tensorflow 2.17.1; keras:  3.7.0; scikit-learn 1.5.2
from datetime import datetime

# As of March 24, 2026 at 8:50 PM: pandas 2.0.0 scikit-learn 1.1.3 tensorflow 2.17.1 keras 3.7.0 tf-keras 2.16.0 numpy 1.26.4 librosa 0.10.1 scipy 1.14.0 plotly 5.23.0 ai_edge_litert 1.3.0 neurokit2 0.2.9

# `Strings & Locations`

In [ ]:
str_accuracy = 'Accuracy'
str_actual_class = 'Actual_class'
str_BOTH_new_fine_tune = 'both_new_layers_fine_tuned_layers'
str_category = 'category'
str_date = 'date'
str_daily_anxiety = 'Daily anxiety'
str_demograpnic_info = 'Demo_info'
str_distance = 'distance'

str_fine_TUNED_layers = 'fine_tuned_layers'
str_filename = 'filename'
str_file_only_new_layers_predict_proba = 'predicted_proba_training_only_new_layers.xlsx' # with_newest_tf_metal_
str_file_predict_prob_fine_tuned_resnet = 'predicted_proba_fine_tuned_ResNet.xlsx'
str_NEW_layers = 'new_layers_only'
str_n_th_iter_of_train = 'n_th_iteration_of_training'
str_on_the_same_day = 'on_the_same_day'

str_predicted_proba = 'Predicted_probability'
str_predicted_class = 'Predicted_class'
str_p_id = 'pid'
str_performance = 'Performance'
str_precision = 'precision'; 
str_recall = 'recall'; 
str_specificity = 'specificity'
str_segment = 'segment'

str_multi_modal = 'multi_modal'
str_tail_of_meta_learner_predict_proba_file = '_including_predicted_class_proba.xlsx'
str_tiles_18 = 'Tiles18'
str_state_anxiety = 'State Anxiety'
str_sapiens = 'SAPIENS'
str_timestamp = 'Timestamp'
str_test_fold = 'test_fold'
str_rqa = '_RQA'

## Evaluation Metrics ###
str_f1_score = 'F1_score'
str_balanced_acc = 'bal_acc'

base_model = 'ResNet_18_RQA'
current_day_status = str_on_the_same_day

if platform.system() == 'Darwin': # Mac == Darwin 🤣🤣🤣
    temp_loc = '... removed the location a bit for privacy .../Documents/SA39/'
else: # Ubuntu currently
    temp_loc = '... removed the location a bit for privacy .../Documents/sa39/'

loc_restrict_EMA_findings = './Results limited EMAs/'

root_loc = os.path.join(temp_loc, 'SAPIENS/')
random_state = 1234

# `Control Center`

In [ ]:
######## Control Center - Start #######
average_approach_for_ev_metric = 'weighted'
crt_split_number = 'splitted_into3'

global_n_minor = 1 # number of samples to be retrieved from the minor class if there is no data of the minor class after selecting the closest 
crt_layer_to_work = str_fine_TUNED_layers # crt: current :)
bool_explore_each_k_fold = False # K-fold altogether of Leave K out cv: Here fold means
bool_include_tiles_meta_data = False # with SAPIENS data :). Could not explore very much this though :).
bool_personalized_meta_learner = True # If it is True, then, it will use all instances (except Test) in the first iteration, in all later stages, there will be at most 6 instances to find the closest 5
bool_global_personalize = False # global personalization. If it is True, the meta learn er will use all participants' data (except Test; to find the closest instances) to predict each state anxiety; Otherwise, it will use the personalized data
bool_add_days = True

# if bool_personalized_meta_learner is False, values of bool_global_personalize and bool_add_days do not matter. However, to keep the filename consistent, I am setting this. Also, it will help me to find the finding file easily to include in the paper.
if bool_personalized_meta_learner is False:
    bool_global_personalize = False 
    bool_add_days = False

bool_save_meta_leaners_predict_proba = True
######## Control Center - End #######

if 'splitted_into3' in crt_split_number:
    n_segments_of_a_day = 3
    teacher_model_fold = 'iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers'
elif 'splitted_into2' in crt_split_number:
    n_segments_of_a_day = 2
    teacher_model_fold = 'iter_5__0a85fd46-fada-434c-9f7a-08b81f9ed8e7__chunk_test_5_val_2__fine_tuned_all_layers' # This was the best teacher model when trained with split 2
else:
    raise ValueError(f'Problem in the string of split number. It can not be {crt_split_number}.')

## WARNING: 🔥 If you change something like modality or a model like not 'ResNet' for SAPIENS, please check the location of TILES-18. Otherwise, there will be a severe error.
loc_tiles_18_perform = os.path.join('... removed the location a bit for privacy .../Documents/SA39/SAPIENS/Daily Anxiety/on_the_same_day/multi_modal/', crt_split_number, 'Tiles18_multi_modal_multi_modal/Performance/ResNet_18')
list_segments = [str_segment + str(temp_nth_segment) for temp_nth_segment in range(1, n_segments_of_a_day+1)]
list_color = ['#FDE0D5', '#F5AE9D', '#df7863', '#CC4D38', '#ac2610', '#6d1002', '#FDE0D5', '#F5AE9D', '#778899', '#696969', '#2F4F4F']

# `Developing Meta Learner`

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, roc_curve
import plotly.graph_objects as go

def clf_report(true_labels, predicted_val_list, bool_print):
    n_digits = 2

    bal_acc = balanced_accuracy_score(y_true=true_labels, y_pred=predicted_val_list)
    precision = precision_score(y_true=true_labels, y_pred=predicted_val_list, average= average_approach_for_ev_metric)
    recall = recall_score(y_true=true_labels, y_pred=predicted_val_list, pos_label=1)
    specificity = recall_score(y_true=true_labels, y_pred=predicted_val_list, pos_label=0)
    f1 = f1_score(y_true=true_labels, y_pred=predicted_val_list, average= average_approach_for_ev_metric)
    acc = accuracy_score(y_true=true_labels, y_pred=predicted_val_list)

    list_columns = [str_precision, str_recall, str_specificity, str_balanced_acc, str_f1_score, 'accuracy'] # WARNING: do NOT change the order of the metrics. It will impact significantly.
    list_performance = [round(precision*100, n_digits), round(recall*100, n_digits), round(specificity*100, n_digits), 
                        round(bal_acc*100, n_digits), round(f1*100, n_digits), round(acc*100, n_digits)] # WARNING: if you change the order of the evaluation metric, please remember to change the order in the list_columns.
    if bool_print:
        print(list_performance, list_columns)
    
    return list_performance, list_columns


def select_n_rows_with_min_minor(df_train_input, single_instance_test_df, list_feature_cols, target_col):
    global n_closest_instances

    if single_instance_test_df.shape[0] > 1:
        raise ValueError('You are calculating the distances based on a single instance\'s data (as you used single_instance_test_df.iloc[0]). So, there can not be more than 1 instance!')

    df_train = df_train_input.copy()
    df_train.reset_index(drop=True, inplace=True)
    df_train[str_distance] = df_train[list_feature_cols].sub(single_instance_test_df.iloc[0][list_feature_cols]).pow(2).sum(axis=1).pow(0.5) # we are already receving the test df containng only features. But to set double safe guard, I am calling only the columns list_feature_cols again for the test df


    df_closest = df_train.nsmallest(n_closest_instances, str_distance)
    list_unique_classes = df_closest[str_actual_class].unique()
    
    if len(list_unique_classes) == 1:
        df_minor = df_train[df_train[target_col] != list_unique_classes[0]].copy()
        df_minor = df_minor[~df_minor.index.isin(df_closest.index)].copy()
        df_additional = df_minor.nsmallest(global_n_minor, str_distance)
        df_closest = pd.concat([df_closest, df_additional], ignore_index=True)

    df_closest.drop(columns=[str_distance], inplace=True) # Do NOT remove this statement. The statement df_minor = df_minor[~df_minor.index.isin(df_closest.index)].copy() will be messed up here 
    df_closest = df_closest.reset_index(drop=True) # new change
    
    # print('select_n_rows_with_min_minor থেকে বলছি|', local_n_minor, Counter(df_close))
    return df_closest


def only_select_n_rows_from_minor_class(df_train_input, single_instance_test_df, list_feature_cols, target_col, major_class_name):
    if single_instance_test_df.shape[0] > 1:
        raise ValueError('You are calculating the distances based on a single instance\'s data (as you used single_instance_test_df.iloc[0]). So, there can not be more than 1 instance!')
    
    # new change block
    df_train = df_train_input.copy()
    df_train.reset_index(drop=True, inplace=True)


    df_train[str_distance] = df_train[list_feature_cols].sub(single_instance_test_df.iloc[0][list_feature_cols]).pow(2).sum(axis=1).pow(0.5)    


    df_n_minor = df_train[df_train[target_col] != major_class_name].copy()
    df_n_minor = df_n_minor.nsmallest(global_n_minor, str_distance)
    df_n_minor.drop(columns=['distance'], inplace=True)
    df_n_minor = df_n_minor.reset_index(drop=True) # new change

    # print('only_select_n_rows_from_minor_class থেকে বলছি|', major_class_name, local_n_minor, Counter(df_n_minor[target_col].tolist()))
    return df_n_minor

def explore_different_meta_learners(list_features, df_data_for_meta):
    # meta_learner(list_features, df_data_for_meta, LogisticRegression(random_state= random_state, class_weight='balanced'), 'LogisticRegression')

    meta_learner(list_features, df_data_for_meta, KNeighborsClassifier(), 'KNeighborsClassifier') # This was the best performing.

    # meta_learner(list_features, df_data_for_meta, DecisionTreeClassifier(random_state= random_state, class_weight='balanced'), 'DecisionTreeClassifier')
    # meta_learner(list_features, df_data_for_meta, RandomForestClassifier(random_state=random_state, class_weight='balanced'), 'RandomForestClassifier')
    ### meta_learner(list_features, df_data_for_meta, LinearSVC(random_state= random_state, class_weight='balanced'), 'LinearSVC') # currently, not using it due to certain issues. Check Root anxiety notebook where you presented models performance
    ### meta_learner(list_features, df_data_for_meta, GradientBoostingClassifier(random_state= random_state), 'GradientBoostingClassifier')
    ### meta_learner(list_features, df_data_for_meta, MLPClassifier(random_state= random_state, hidden_layer_sizes=2, early_stopping=True), 'MLPClassifier') # sometimes, there remains a single instance of a class. However, MLP needs at least 2 instances of each class.
    ### meta_learner(list_features, df_data_for_meta, XGBClassifier(random_state = random_state), 'XGBClassifier')

    # # stacking_classifier = StackingClassifier(estimators=[('DT', DecisionTreeClassifier(random_state= random_state, class_weight='balanced',  max_features=2, max_depth=2, min_samples_leaf=2)),
    # #                                                      ('LSC', RandomForestClassifier(random_state=random_state, class_weight='balanced', max_features=2, max_depth=2, n_estimators=10))],
    # #                                                      final_estimator=LogisticRegression(random_state= random_state, class_weight='balanced', C=10))
    # # return meta_learner(list_features, df_data_for_meta, stacking_classifier, 'ensemble_classifier')


def check_whether_features_are_expected(list_features):
    for __temp_col in list_features:
        if str_segment not in __temp_col:
            raise ValueError('Severe problem! As per the current logic, there should not be any column other than segment.... in features set')

def meta_learner(list_features, df_data_for_meta, root_model, model_name):
    global n_closest_instances, loc_performances, dict_clf_true_class_list, dict_clf_predict_proba_list
    global inject_m_EMAs, pick_from_first_t_instances, current_ds
    global list_length_emas, limit_n_days

    global list_df_all_models_windows_perf, window_folder, multi_mode_comb, list_df_all_models_windows_columns
    list_df_all_models_windows_columns = []
    list_clf_no_predict_proba = ['NearestCentroid', 'LinearSVC'] # These classifiers do not have a predicted probability by defaults

    check_whether_features_are_expected(list_features) # checking whether the features contain only the predicted probabilities for the segments only.

    if df_data_for_meta.isna().sum().sum() > 0:
        raise ValueError("Ideally, there should not be any NaN values considering currently, we are excluding the segment instances for which we do not have the predicted probabilities of all segments!")
    
    list_pid = []
    list_ts = [] # ts: timestamp
    list_actual_class = []
    list_predicted_class = []
    list_predicted_proba = []



    for pid in sorted(df_data_for_meta[str_p_id].unique()):
        test_df_meta_learner = df_data_for_meta[df_data_for_meta[str_p_id] == pid].copy()
        root_train_data_meta_learner = df_data_for_meta[df_data_for_meta[str_p_id] != pid].copy()
        personal_train_df = root_train_data_meta_learner.copy()
        list_test_ts = sorted(test_df_meta_learner[str_timestamp].tolist()) # ts: timestamp

        if bool_include_tiles_meta_data and (str_sapiens == current_ds):
            df_tiles_18_new_layers = pd.read_excel(os.path.join(loc_tiles_18_perform, str_file_only_new_layers_predict_proba))
            df_tiles_18_fine_tune_layers = pd.read_excel(os.path.join(loc_tiles_18_perform, str_file_predict_prob_fine_tuned_resnet))
            if crt_layer_to_work == str_NEW_layers:
                personal_train_df = pd.concat([personal_train_df, df_tiles_18_new_layers])
            elif crt_layer_to_work == str_fine_TUNED_layers:
                personal_train_df = pd.concat([personal_train_df, df_tiles_18_fine_tune_layers])
            # # # # elif crt_layer_to_work == str_BOTH_new_fine_tune:
            # # # #     df_tiles_18_new_layers.rename(columns = {str_period +'__'+str_0_11: str_file_only_new_layers_predict_proba+str_0_11, str_period +'__'+str_12_23: str_file_only_new_layers_predict_proba+ str_12_23}, inplace=True)
            # # # #     df_tiles_18_fine_tune_layers.rename(columns = {str_period +'__'+str_0_11: str_file_predict_prob_fine_tuned_resnet+str_0_11, str_period +'__'+str_12_23: str_file_predict_prob_fine_tuned_resnet+ str_12_23}, inplace=True)
                
            # # # #     df_tiles_18_both_new_fine_tuned =  pd.merge(df_tiles_18_new_layers, df_tiles_18_fine_tune_layers, on=[str_timestamp, str_p_id], how='inner')
            # # # #     df_tiles_18_both_new_fine_tuned.rename(columns={'Actual_class_x': str_actual_class}, inplace=True)
            # # # #     personal_train_df = pd.concat([personal_train_df, df_tiles_18_both_new_fine_tuned])
            # # # #     personal_train_df.drop(columns=['Actual_class_y'], inplace=True)
            
            if personal_train_df.isna().sum().sum() > 0:
                nan_rows = personal_train_df[personal_train_df.isna().any(axis=1)]
                display(HTML(nan_rows.to_html()))
                raise ValueError('There can not be any NaN values')

        if len(list_test_ts) != len(set(list_test_ts)):
            print(f'This {pid} has duplicate dates which should not be! {list_test_ts}')
            raise ValueError('\n\n There should not be any duplicate timestamps in case of any participant! \n\n')
        
        single_p_list_actual_class = []
        single_p_list_predicted_class = []
        
        # new change block
        sdf_ALL_ema = pd.DataFrame()
        set_date = set()
        bool_1_warm_up_NOT_done = True

        if bool_personalized_meta_learner:
           
            for single_test_ts in list_test_ts:
                single_test_data_instance = test_df_meta_learner[test_df_meta_learner[str_timestamp] == single_test_ts].copy()

                # Working on timestamps for date
                # Checked manually (what is the date we see from a timestamp and whether it makes sense(whether it is within study period)) both (TILES & SAPIENS) date finding process. 
                if current_ds == str_sapiens:
                    single_test_data_instance[str_date] = pd.to_datetime(single_test_data_instance[str_timestamp], unit = 's', utc = True).dt.tz_convert(pytz.timezone('US/Eastern')).dt.date
                elif current_ds == str_tiles_18:
                    single_test_data_instance[str_date] = pd.to_datetime(single_test_data_instance[str_timestamp], unit = 's', utc = True).dt.tz_convert(pytz.timezone('America/Los_Angeles')).dt.date 
                    # For TILES, I checked time zone of each EMA completed date-time. There were only {'08:00', '07:00'} time zones, which corresponds to America/Los_Angeles [checked wikipedia].
                    # Time zone is so critical :). Made a mistake in 2024 :(
                else: raise ValueError('What is this dataset, Sabbir? I do not know.')

                if bool_global_personalize:
                    personal_train_df = select_n_rows_with_min_minor(df_train_input = root_train_data_meta_learner, single_instance_test_df = single_test_data_instance[list_features], 
                                                                     list_feature_cols = list_features, target_col = str_actual_class)
                else:
                    personal_train_df = select_n_rows_with_min_minor(df_train_input = personal_train_df, single_instance_test_df = single_test_data_instance[list_features], 
                                                                     list_feature_cols = list_features, target_col = str_actual_class)


                # new change block
                if bool_add_days:

                    set_date.update(single_test_data_instance[str_date].tolist())
                    n_dates = len(set_date)
                    
                    if n_dates <= limit_n_days:
                        sdf_ALL_ema = pd.concat([sdf_ALL_ema, single_test_data_instance]) # not using all previous EMAs currently. Depending on the conditions and values like value of inject_m_EMAs, a number of EMAs will be used, when inject_m_EMAs  is > 0 
                        sdf_ALL_ema = sdf_ALL_ema.reset_index(drop=True)

                    sdf_INJECT_ema = sdf_ALL_ema.copy()
                    crt_n_EMAs_available = len(sdf_INJECT_ema) # I double thought whether checking len(sdf_ALL_ema) may cause information leakage. I believe it is not, since I am just checking whether there is any EMA and if any, what is the length. User may or may not provide any EMA
                    
                    # inject_m_EMAs > 0: we need to explore how the algo performs without injecting any EMAs and hence, set the condition inject_m_EMAs > 0 so that inject_m_EMAs==0 helps not to run this block and we can explore without injecting any EMAs.
                    
                    if (inject_m_EMAs > 0) and (n_dates > limit_n_days ) and (crt_n_EMAs_available >= inject_m_EMAs) and bool_1_warm_up_NOT_done: # you can set None to inject_m_EMAs, if you do not want to inject any EMA
                        
                        list_length_emas.append(len(sdf_INJECT_ema)) # just to check how many EMAs were there for each P, usually. It helped to debug as well :)
                        
                        if len(sdf_INJECT_ema) > inject_m_EMAs:
                            sdf_INJECT_ema = sdf_INJECT_ema.sample(n= inject_m_EMAs, random_state = random_state)


                        if inject_m_EMAs < n_closest_instances: # means we can add pseudo-labelled (i.e., predicted class labeled) instances here as well. If the training data set in personal_train_df is lower than 5, there will be an error in KNN, since we are using the default neighbors k 5 in sklearn. Besides, we are exploring various closest number of samples like 5, 10, 15, 20, etc., and so, we need the minimum number of samples as defined by the value of n_closest_instances
                            # Inside select_n_rows_with_min_minor(), I am resetting the index and so, there should not be any problem with the usage of index to select and drop rows afterwards.
                            str_is_injected = '_is_injected'
                            sdf_INJECT_ema[str_is_injected] = True
                            personal_train_df[str_is_injected] = False
                            
                            ## 🐝🐝🐝🐝🐝 WARNING: please, do not keep sdf_INJECT_ema at the begining of the list while concating. Otherwise, the drop duplicates operation, where we are keeping the last rows (i.e., injected EMAs) (for model by  keep = 'last'), will be replaced by the pseudo labelled/predicted class labelled data.
                            personal_train_df = pd.concat([personal_train_df, sdf_INJECT_ema]).reset_index(drop = True) # always reset please :). Otherwise, index based operations will vanish you :(
                            personal_train_df = personal_train_df.drop_duplicates(subset = [str_p_id, str_timestamp], keep = 'last') # Dropping duplicates (if any) to priortaize the user defined labels. Here, timestamp presents the time when the model predicted state anxiety. In real-world deployment, an EMA will be sent at the same time of predicting. Besides, here I kept str_p_id for future reusibility if needed. Here, no impact, since there is a single P.
                            n_excess_emas = len(personal_train_df) - n_closest_instances # excess EMAs if any.

                            if n_excess_emas > 0:
                                mask_not_inject = personal_train_df[str_is_injected] == False # if excess, dropping from pseudo labelled (i.e., predicted class) instances, prioratizing user provided EMAs/labels
                                index_rand_drop = personal_train_df[ mask_not_inject ].sample(n_excess_emas, random_state = random_state).index
                                personal_train_df = personal_train_df.drop(index_rand_drop)
                            
                            ## Dropping things and making sanity check
                            personal_train_df.drop(columns = [str_is_injected], inplace = True)
                            personal_train_df.reset_index(drop = True, inplace = True)
                            if personal_train_df.duplicated(subset=[str_p_id, str_timestamp]).any(): raise ValueError('Hey Sabbir, how can still be such duplicated instances? Problem 😡') ## Sanity check :)
                            
                        else: # the number of EMAs I am going to inject, is same as I have currently (they are not user provided though) in training. So, drop all to get user labelled EMAs. But it is going to happen once only and so, it is like giving a big push the meta learner (anything better to describe it 🤔? :) ).
                            personal_train_df = sdf_INJECT_ema.copy()


                        # If there is a data of a single class only after injecting EMAs, getting data of minor class from global training pool.
                        if len(personal_train_df[str_actual_class].unique()) == 1:
                            df_n_closest = only_select_n_rows_from_minor_class(df_train_input = root_train_data_meta_learner, 
                                                                               single_instance_test_df= single_test_data_instance[list_features],   
                                                                               list_feature_cols = list_features, 
                                                                               target_col = str_actual_class, 
                                                                               major_class_name = personal_train_df.iloc[0][str_actual_class])
                            personal_train_df = pd.concat([personal_train_df, df_n_closest])
                        

                        personal_train_df.reset_index(drop=True, inplace = True)
                        bool_1_warm_up_NOT_done = False

                x_train = personal_train_df[list_features].values.tolist()
                y_train = personal_train_df[str_actual_class].tolist()
                x_test = single_test_data_instance[list_features].values.tolist()

                ML_model = deepcopy(root_model) # just to be cautious in case in future I include a model which does not reinitialize everything after calling fit(); as a safe guard; I know every time we call fit(), we are inititalizing again everything
                ML_model.fit(x_train, y_train)
                list_pid.append(pid)
                list_ts.append(str(single_test_ts))
                list_actual_class.extend(single_test_data_instance[str_actual_class].tolist()) # single_test_data_instance actually contains a single instance's values as we are moving one state anxiety by one
                list_predicted_class.extend(ML_model.predict(x_test))
                if model_name not in list_clf_no_predict_proba:
                    list_predicted_proba.extend(ML_model.predict_proba(x_test)[:, 1]) #  # for class 1 only (i.e., class presenting the presence of state anxiety).

                single_p_list_actual_class.extend(single_test_data_instance[str_actual_class].tolist())
                single_p_list_predicted_class.extend(ML_model.predict(x_test))

                if bool_add_days:
                    _1_test_to_train = deepcopy(single_test_data_instance) # always copy so that you can use the instance independently :)
                    _1_test_to_train[str_actual_class] = ML_model.predict(x_test)[0] # Changing the actual class for next training/prediction to reflect the real-world deployment scenario when we may not have the ground-truth/user provided labels always. If there is, we can inject (as written codes above) as a warm start
                    if len(ML_model.predict(x_test)) > 1: raise ValueError('Severe problem. It can not happen, Sabbir! 😡') # Sanity check.

                    # Double checked the places where personal_train_df is used and whether changes in actual class will have any impact anything (e.g., evaluation) other than training related things. Did not find.
                    personal_train_df = pd.concat([personal_train_df, _1_test_to_train])

                    

        else:
            ML_model = deepcopy(root_model)
            x_test = test_df_meta_learner[list_features].values.tolist()

            ML_model.fit(root_train_data_meta_learner[list_features].values.tolist(), 
                         root_train_data_meta_learner[str_actual_class].tolist())

            list_pid.extend(np.repeat(pid, test_df_meta_learner.shape[0]))
            list_ts.extend(test_df_meta_learner[str_timestamp].tolist())
            list_actual_class.extend(test_df_meta_learner[str_actual_class].tolist())
            list_predicted_class.extend(ML_model.predict(x_test))
            if model_name not in list_clf_no_predict_proba:
                list_predicted_proba.extend(ML_model.predict_proba(x_test)[:, 1]) # for class 1 only.

            print(model_name, pid,  clf_report(true_labels= test_df_meta_learner[str_actual_class].tolist(), predicted_val_list= ML_model.predict(x_test), bool_print= False))

    if bool_save_meta_leaners_predict_proba and (model_name not in list_clf_no_predict_proba) and (not bool_explore_each_k_fold):
        print(len(list_pid), len(list_ts), len(list_predicted_class), len(list_predicted_proba))
        df_meta_includ_predict_proba_class = pd.DataFrame({str_p_id: list_pid, str_timestamp: list_ts, 
                                                           str_predicted_class: list_predicted_class, str_predicted_proba: list_predicted_proba})
    
        df_meta_includ_predict_proba_class[str_timestamp] = df_meta_includ_predict_proba_class[str_timestamp].astype(str)
        df_data_for_meta[str_timestamp] = df_data_for_meta[str_timestamp].astype(str)

        df_meta_includ_predict_proba_class = pd.merge(df_data_for_meta, df_meta_includ_predict_proba_class, on=[str_p_id, str_timestamp], how='inner')
        df_meta_includ_predict_proba_class.sort_values(by=[str_p_id, str_timestamp], inplace=True)
        df_meta_includ_predict_proba_class.reset_index(inplace=True, drop=True)
        
        if df_meta_includ_predict_proba_class.shape[0] != df_data_for_meta.shape[0]:
            raise ValueError(f'Shape of df_meta_includ_predict_proba_class and df_data_for_meta shold be same! Currently, df_meta_includ_predict_proba_class: {df_meta_includ_predict_proba_class.shape} and df_data_for_meta: {df_data_for_meta.shape}')

        ## print('🐝🐝🐝🐝🐝 Saving per p performance to ', loc_performances)
        ## df_meta_includ_predict_proba_class.to_excel(os.path.join(loc_performances, base_model +'__'+ crt_layer_to_work+'__'+current_ds+'__'+current_day_status+'__'+model_name+ '_incl_predicted_instance_'+ str(bool_personalized_meta_learner)+ '_global_person_' + str(bool_global_personalize) + '_add_prev_days_'+ str(bool_add_days)+ str_tail_of_meta_learner_predict_proba_file), index=False) # we do not need to include bool_explore_each_k_fold since bool_save_meta_leaners_predict_proba will be false when bool_explore_each_k_fold will be true 

    # print('roc_auc_score: ', roc_auc_score(y_true= list_actual_class, y_score=list_predicted_proba, average=average_approach_for_ev_metric))

    dict_clf_true_class_list[model_name] = list_actual_class
    dict_clf_predict_proba_list[model_name] = list_predicted_proba

    list_performance, list_columns = clf_report(true_labels= list_actual_class, predicted_val_list= list_predicted_class, bool_print= False)
    details_single_models_performance = list_performance.copy()
    details_single_models_performance.extend([window_folder, multi_mode_comb, n_closest_instances, global_n_minor,  model_name, n_segments_of_a_day, len(list_actual_class), inject_m_EMAs, pick_from_first_t_instances])
    list_df_all_models_windows_columns.extend(list_columns)
    list_df_all_models_windows_columns.extend(['window_folder', 'multi_mode_comb', 'n_closest_instances', 'global_n_minor',  'model_name', 'n_segments_of_a_day', '# of EMAs', 'inject_m_EMAs', 'pick_EMAs_from_first_t_instances'])

    list_df_all_models_windows_perf.append(details_single_models_performance)
    print(model_name, dict(zip(list_columns, list_performance)))

    return list_performance, list_columns


print(datetime.now())

## `Starting Point`

In [ ]:
def get_file_names_without_segment_name(list_file_names):
    set_file_names_excl_segment_name = set()
    
    for temp_file_name in list_file_names:
        # We can optimize it later by removing the for loop :). Note: the codes are fine. A file can have maximum 1 segment name. Right? If yes, why do we need to iterate over each segment which will increase the time complexity n_segments_of_a_day times. Just look into the file name and replace just the segment name with ''
        for crt_segment in list_segments:
            temp_file_name = temp_file_name.replace(crt_segment, '')
        set_file_names_excl_segment_name.add(temp_file_name)

    return set_file_names_excl_segment_name


def prepare_weakly_predict_prob_for_meta(df_data):
    dict_file_prob = dict(zip(df_data[str_filename].tolist(), df_data[str_predicted_proba].tolist()))
    set_file_names_excl_segment_name = get_file_names_without_segment_name(df_data[str_filename].tolist())

    pred_prob_data_list_segments_df = []

    # This loop is used to get the predicted probabilities from only the files (i.e., files of a date of a pid) for which there are data of all periods of the day
    # Finally, we will have a dataframe like probability_of_segment_1_image - probability_from_segment_2_image - ..... - class - pid - date - n_th_iteration_of_training (like fold) 
    for filename_without_seg_name in set_file_names_excl_segment_name:
        single_list_pred_prob = []
        bool_all_periods = True # Adding the data only when there are data of both periods (if there are at least min_number_of_samples_for_image (Jan. 5, 2024: 50 samples))
        
        for segment in list_segments:
            splitted_file_name = filename_without_seg_name.split('____') # Let's say, an image file name is "0__413__2024-03-06__segment1__Battery__Level.png". When you remove the segment name, the file name will become "0__413__2024-03-06____Battery__Level" (i.e., will contain ____)
            filename_with_segment_name = splitted_file_name[0] +'__'+  segment +'__'+ splitted_file_name[1]
            
            if (filename_with_segment_name in dict_file_prob.keys()) and bool_all_periods:
                single_list_pred_prob.append(dict_file_prob.get(filename_with_segment_name))
            else:
                print(f'This segment was not included during meta learner development: {filename_with_segment_name}')
                bool_all_periods = False

        if bool_all_periods:
            # safeguard
            if df_data[df_data[str_filename] == filename_with_segment_name].shape[0] > 1:
                raise ValueError(f"Severe problem! There can not be more than 1 row with the same file name (where segment name is included) {filename_with_segment_name}. \
                                 If there are more than 1 rows, then the class, pid, and timestamp we are retrieving in the following statement by iloc[0] is wrong since we are retreving only the first one!")

            single_list_pred_prob.extend([df_data[df_data[str_filename] == filename_with_segment_name][str_actual_class].iloc[0], 
                                          df_data[df_data[str_filename] == filename_with_segment_name][str_p_id].iloc[0], 
                                          df_data[df_data[str_filename] == filename_with_segment_name][str_timestamp].iloc[0]])
            pred_prob_data_list_segments_df.append(single_list_pred_prob)
        
    list_columns = list_segments.copy()
    list_columns.extend([str_actual_class, str_p_id, str_timestamp])

    return pd.DataFrame(data= pred_prob_data_list_segments_df, columns = list_columns)


def get_updated_column_names(list_column_names, str_model_type):
    dict_updated_col_names = dict()
    for col_name in list_column_names:
        if col_name not in [str_actual_class, str_timestamp, str_p_id]:
            dict_updated_col_names[col_name] = str_model_type +'_'+ col_name

    return dict_updated_col_names


# preparing the dataframes for the meta learner. 
def prepare_dfs_and_eval_based_on_meta_learner(loc_predict_proba_new_layers, loc_predict_proba_fine_tuned_all_layers):
    global loc_performances
    
    if os.path.exists(loc_predict_proba_new_layers):
        df_data_for_meta_new_layers = pd.read_excel(loc_predict_proba_new_layers)
    else:
        print('\n\n\n Remember, did not get the dataframe containing new layers performance! \n\n\n')
        
    
    if crt_layer_to_work != str_NEW_layers: # i.e., it can be a fine tuned layer or exploration of both layers
        df_data_for_meta_fine_tuned = pd.read_excel(loc_predict_proba_fine_tuned_all_layers)

    if bool_explore_each_k_fold: # when bool_explore_each_k_fold is True, it means we are exploring the raw predicted probabilities where each row contains probabilities of a SINGLE segment only. So, we need to prepare dataframe containing columns like segment1, segment2, ...., pid, actual class, ...
        df_data_for_meta_new_layers = prepare_weakly_predict_prob_for_meta(df_data_for_meta_new_layers)
        df_data_for_meta_fine_tuned = prepare_weakly_predict_prob_for_meta(df_data_for_meta_fine_tuned)
    else:
        if crt_layer_to_work == str_NEW_layers:
            print('Shape of the dataset for new layers', df_data_for_meta_new_layers.shape, loc_performances)

        if crt_layer_to_work == str_fine_TUNED_layers:
            print('Shape of the dataset for fine tuned layers', df_data_for_meta_fine_tuned.shape, loc_performances) 
        
    if crt_layer_to_work  == str_BOTH_new_fine_tune:
        df_data_for_meta_new_layers.rename(columns = get_updated_column_names(df_data_for_meta_new_layers.columns.tolist(), str_file_only_new_layers_predict_proba), inplace=True)
        df_data_for_meta_fine_tuned.rename(columns = get_updated_column_names(df_data_for_meta_fine_tuned.columns.tolist(), str_file_predict_prob_fine_tuned_resnet), inplace=True)
        
        df_data_for_meta = pd.merge(df_data_for_meta_new_layers, df_data_for_meta_fine_tuned, on=[str_timestamp, str_p_id], how='inner')
        df_data_for_meta.reset_index(inplace=True, drop=True)
        df_data_for_meta.rename(columns={'Actual_class_y': str_actual_class}, inplace=True)
        df_data_for_meta.drop(columns=['Actual_class_x'], inplace=True)

        list_features = [col for col in df_data_for_meta.columns.tolist() if str_segment in col] # using only the columns containing the predicted probabilities.
    else:
        list_features = list_segments.copy()
        if crt_layer_to_work == str_NEW_layers:
            df_data_for_meta = df_data_for_meta_new_layers.copy()
        elif crt_layer_to_work == str_fine_TUNED_layers:
            df_data_for_meta = df_data_for_meta_fine_tuned.copy()
        else:
            raise ValueError(f'Please, check the value of crt_layer_to_work variable. It should not be {crt_layer_to_work}')
    
    if str_date in df_data_for_meta.columns.tolist():
        df_data_for_meta.rename(columns={str_date: str_timestamp}, inplace=True)
    
    explore_different_meta_learners(list_features= list_features, df_data_for_meta= df_data_for_meta)


def iterate_through_predict_proba_files():
    global loc_performances

    # Strategies :)
    # 1. Get the location of each fine_tuned predicted probability file. Also, get the corresponding new layers' predicted probability 
    # 2. pass the locations to prepare_dfs_and_eval_based_on_meta_learner
    # 3. get the dataframes of fine tuned and new layers' predicted probabilities
    # 4. prepare whole window level predicted probabilities (that is predicted probability of period segment 1 and segment 2)....
    # 5. Depending on the conditions (see Control Center :)), the models will developed

    if bool_explore_each_k_fold:
        for perf_file in os.listdir(loc_performances):
            if ('DS_Store' in perf_file) or (str_file_predict_prob_fine_tuned_resnet == perf_file) or (str_file_only_new_layers_predict_proba == perf_file)\
                  or os.path.isdir(os.path.join(loc_performances, perf_file)) or perf_file.endswith('.txt') or (str_tail_of_meta_learner_predict_proba_file in perf_file):
                continue
            
            if str_file_only_new_layers_predict_proba in perf_file: # restricting it so that we do not explore the same file multiple times. This will ensure that we are reading new layers and fine tuned files of 1 fold at a time. Below, you will see that we are reading both files by replacing the fine tuned layer name with new layer 
                continue
            # perf_file  basically contains the predicted probabilities of each k-fold
            
            loc_perf_NEW_Layers_file = os.path.join(loc_performances, perf_file.split('__')[0] +'__'+ str_file_only_new_layers_predict_proba)
            loc_perf_TUNED_layers_file = os.path.join(loc_performances, perf_file) # as we are not running these codes while str_file_only_new_layers_predict_proba is in file, it can be ensured that we are getting str_file_predict_prob_fine_tuned_resnet while running these codes.
            
            if 'including_predicted_class' in loc_perf_TUNED_layers_file:
                raise ValueError('This should not run! If it runs, I will have to understand why is that')

            prepare_dfs_and_eval_based_on_meta_learner(loc_predict_proba_new_layers = loc_perf_NEW_Layers_file, 
                                                       loc_predict_proba_fine_tuned_all_layers = loc_perf_TUNED_layers_file)
    else:
        prepare_dfs_and_eval_based_on_meta_learner(loc_predict_proba_new_layers = os.path.join(loc_performances, str_file_only_new_layers_predict_proba), 
                                                   loc_predict_proba_fine_tuned_all_layers = os.path.join(loc_performances, str_file_predict_prob_fine_tuned_resnet))

## SAPIENS

In [ ]:
#  windows for state anxiety; like window of 1, 1.5 hour, 2 hours ....

dict_clf_true_class_list = dict()
dict_clf_predict_proba_list = dict()

def explore_windows_and_multi_modal_models_peformance():
    global multi_mode_comb, window_folder, list_df_all_models_windows_perf, loc_performances, n_closest_instances, current_ds
    global inject_m_EMAs, pick_from_first_t_instances, list_length_emas, limit_n_days
    current_ds = str_sapiens
    list_df_all_models_windows_perf = []
    list_length_emas = []
    limit_n_days = 1 ## End of Check

    for window_folder in sorted(os.listdir(os.path.join(root_loc, str_state_anxiety)), reverse=True):
        if ('DS_Store' in window_folder) or ('3.5_hours' in window_folder) or ('ds_do_NOT_deletev' in window_folder) or ('hours' not in window_folder):
            continue
        
        if '2.5_hours' not in window_folder: # working on 2.5 hours only, since this is the best performing window
            continue

        loc_multi_mode_combination = os.path.join(root_loc, str_state_anxiety, window_folder, str_multi_modal, crt_split_number)
        for multi_mode_comb in sorted(os.listdir(loc_multi_mode_combination)):
            if 'DS_Store' in multi_mode_comb or (multi_mode_comb not in 'SAPIENS_HeartRate_Light__accel___magnet_'):
                continue
            
            loc_performances = os.path.join(loc_multi_mode_combination, multi_mode_comb, str_performance, 'ResNet_18', teacher_model_fold) # sampling_rate_1_ResNet_18, ... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/1.5_hours_1_vs_2_10_split/multi_modal/splitted_into3/SAPIENS_HeartRate_Light__accel___magnet_/Performance/ResNet_18/iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers
            
            if limit_n_days == 1: 
                max_limit_of_injection = 5 # If there is a day, there can be maximum 7 EMAs (since Sensus is supposed to send max 7 EMAs/day). Then, how can we sample more EMAs like 8. And also, it is unlikely that participants were 100% compliant. Also, to be consistent with TILES exploration,... just restricted upto 5 EMAs.
            else:
                max_limit_of_injection = 10

            for inject_m_EMAs in range(0, max_limit_of_injection+1, 1):
                pick_from_first_t_instances = inject_m_EMAs

                for n_closest_instances in range(5, 10, 5): # Exploring only n_closest_instances 5, since it was the best performing with all historical EMA based model.
                    print('Exploring: ', window_folder, multi_mode_comb, n_closest_instances, 'inject_m_EMAs:', inject_m_EMAs, 'pick from n: ', pick_from_first_t_instances)
                    iterate_through_predict_proba_files()

explore_windows_and_multi_modal_models_peformance()
print(datetime.now())

## `Saving Findings`

In [ ]:
def saving_models_performances():
    global inject_m_EMAs, pick_from_first_t_instances, current_ds, limit_n_days
    
    special_note = f'sample_from_past{limit_n_days}DaysOnly_'

    file_name_to_save_models_findings = current_ds +'_'+ special_note +\
        '_inclTiles_' +str(bool_include_tiles_meta_data)+ '_kFold' +str(bool_explore_each_k_fold)+ '_inclPredict_'+ str(bool_personalized_meta_learner) +'_'+ crt_split_number +'_'+ '_globPerson_' + str(bool_global_personalize) + '_preDay_'+ str(bool_add_days)+ '.xlsx'
    df_all_models_perf = pd.DataFrame(data= list_df_all_models_windows_perf, columns= list_df_all_models_windows_columns)
    df_all_models_perf.to_excel(os.path.join(loc_restrict_EMA_findings, file_name_to_save_models_findings), index=False) # '1st_'

saving_models_performances()
print(datetime.now())

## ROC-AUC Curve for SAPIENS

In [ ]:
def process_clf_name(clf):
    return clf.replace('Classifier', '').replace('LogisticRegression', 'Logit').replace('KNeighbors', 'KNN')

def plot_pr_curve_plotly(title="Precision–Recall Curve"):
    global  dict_clf_true_class_list, dict_clf_predict_proba_list
    fig = go.Figure()

    for index, clf_name in enumerate(sorted(dict_clf_true_class_list.keys())):
        if clf_name == 'LinearSVC':
            continue

        y_true = np.asarray(dict_clf_true_class_list.get(clf_name))
        y_proba = np.asarray(dict_clf_predict_proba_list.get(clf_name))
        
        precision, recall, _ = precision_recall_curve (y_true, y_proba)
        ap = average_precision_score(y_true, y_proba, average= average_approach_for_ev_metric)
        fig.add_trace(go.Scatter(x=recall, y=precision, mode="lines", line=dict(color=list_color[index]), marker=dict(color="#000000"), name=f"{process_clf_name(clf_name)}: {ap*100:.2f}"))

    fig.update_layout(
        title=title,
        xaxis_title="Sensitivity",
        yaxis_title="Precision",
        plot_bgcolor="white",
        font=dict(family="Times New Roman", size=18),
        margin=dict(l=20, r=20, t=40, b=40),
        showlegend=True)
    fig.update_layout(legend=dict(x=0.5, y=0,  xanchor="center", yanchor="bottom"))
    fig.update_xaxes(range=[0, 1], showline=True, mirror=True, linewidth=1)
    fig.update_yaxes(range=[0, 1.05], showline=True, mirror=True, linewidth=1)
    fig.show()


def plot_roc_curve_plotly(title="ROC Curve"):
    global  dict_clf_true_class_list, dict_clf_predict_proba_list
    fig = go.Figure()

    for index, clf_name in enumerate(sorted(dict_clf_true_class_list.keys())):
        if clf_name == 'LinearSVC':
            continue

        y_true = np.asarray(dict_clf_true_class_list.get(clf_name))
        y_proba = np.asarray(dict_clf_predict_proba_list.get(clf_name))

        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc_val = roc_auc_score(y_true, y_proba, average=average_approach_for_ev_metric)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", line=dict(color=list_color[index]), marker=dict(color="#000000"), name=f"{process_clf_name(clf_name)}: {auc_val*100:.2f}"))

    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="markers+lines",
        name="Random guess",
        line=dict(color="#CC4D38", dash="dash"), marker=dict(color="#000000"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="False Positive Rate (1 − Specificity)",
        yaxis_title="True Positive Rate (Sensitivity)",
        plot_bgcolor="white",
        font=dict(family="Times New Roman", size=18),
        margin=dict(l=20, r=20, t=40, b=40),
        showlegend=True
    )
    fig.update_layout(legend=dict(x=0.5, y=0,  xanchor="center", yanchor="bottom"))
    fig.update_xaxes(range=[0, 1], showline=True, mirror=True, linewidth=1)
    fig.update_yaxes(range=[0, 1], showline=True, mirror=True, linewidth=1)
    fig.show()


plot_roc_curve_plotly()
plot_pr_curve_plotly()

# `Exploring TILES-18`

In [ ]:
#  windows for state anxiety; like window of 1, 1.5 hour, 2 hours ....

dict_clf_true_class_list = dict()
dict_clf_predict_proba_list = dict()

def explore_tiles18_peformance():
    global multi_mode_comb, window_folder, list_df_all_models_windows_perf, loc_performances, n_closest_instances, current_ds, bool_include_tiles_meta_data
    global inject_m_EMAs, pick_from_first_t_instances, limit_n_days
    global list_length_emas
    list_df_all_models_windows_perf = []
    multi_mode_comb = 'TILES_18_modalities'
    window_folder = 'TILES_18_windows'
    current_ds = str_tiles_18
    bool_include_tiles_meta_data = False
    list_length_emas = []
    limit_n_days = 5 ## End of Check

    if limit_n_days == 5:
        max_limit_of_injection = 5 # If there are 5 days, there can be 5 EMAs. Then, how can we randomly sample more EMAs like 6 :). We can NOT. Hence, we restricted up to 5 :)
    else:
        max_limit_of_injection = 10

    

    loc_multi_mode_combination = os.path.join(root_loc, str_daily_anxiety, str_on_the_same_day, str_multi_modal, crt_split_number)
    loc_performances = os.path.join(loc_multi_mode_combination, 'Tiles18_multi_modal_multi_modal', str_performance, 'ResNet_18') # ... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/1.5_hours_1_vs_2_10_split/multi_modal/splitted_into3/SAPIENS_HeartRate_Light__accel___magnet_/Performance/ResNet_18/iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers

    for inject_m_EMAs in range(0, max_limit_of_injection+1, 1):
        pick_from_first_t_instances = inject_m_EMAs

        for n_closest_instances in list(range(5, 10, 5)): # last point is stop-step
            print('Exploring: ', n_closest_instances, 'inject_m_EMAs:', inject_m_EMAs, 'pick from n: ', pick_from_first_t_instances)
            iterate_through_predict_proba_files()

    
    special_note = f'sample_from_past{limit_n_days}DaysOnly_'

    file_name_to_save_models_findings =  'TILES_'+ special_note +\
        '_K_fold'+str(bool_explore_each_k_fold)+ '_incl_predictInstance_'+ str(bool_personalized_meta_learner) +'_'+ crt_split_number +'_'+ '_globePerson_' + str(bool_global_personalize) + '_prevDay_'+ str(bool_add_days)+ '.xlsx'
    df_all_models_perf = pd.DataFrame(data= list_df_all_models_windows_perf, columns= list_df_all_models_windows_columns)
    df_all_models_perf.to_excel(os.path.join(loc_restrict_EMA_findings, file_name_to_save_models_findings), index=False)

explore_tiles18_peformance()
print(datetime.now())

# Presenting the findings without or limited EMA

In [ ]:
str_n_neighbours = 'k neighbors'

# Checked: 1
def df_to_full_latex_table(df, caption="", label="", round_upto_n_decimal=2):
    fmt = "{:."+str(round_upto_n_decimal)+"f}"
    k_values = df.columns.get_level_values(1).unique() # get_level_values(1) returns the values of nearest neighbours used in KNN. level 0: str_n_neighbour as string, presenting what the values of level 1 presents.
    
    ######## Getting dynamic column format
    cols_per_k = len(df.columns) // len(k_values)    
    dynamic_col_format = "c|" + ("c" * cols_per_k + "|") * len(k_values)

    ######## Iterating through each row and formatting the values.
    for idx, row in df.iterrows():
        best_k = max(k_values, key=lambda k: row[(str_n_neighbours, k, 'BA')])
        
        for col in df.columns:
            top_level, k, metric = col
            str_val = fmt.format(row[col])
            
            if k == best_k:
                df.at[idx, col] = str_val
            else:
                df.at[idx, col] = str_val

    df.reset_index(inplace=True) # need to reset index to see the number of EMAs for example, 0, 1, 2, 3, 4, 5. Double checked.
    
    
    ######## Working for latex
    tabular = df.to_latex(index=False, 
                          column_format = dynamic_col_format,
                          multicolumn_format="c") # c means center :)
    
    
    # The following commented line is to set partial horizontal line :). Simply, put it below \multicolumn{5}{c}{3} & \multicolumn{5}{c}{4} & \multicolumn{5}{c}{5}
    # \cmidrule(lr){2-6} \cmidrule(lr){7-11} \cmidrule(lr){12-16}

    tabular = tabular.replace("N EMAs", "\\multirow{3}{*}{N EMAs}") # to keep {N EMAs} at the center horizontally and vertically. 

    return (
        "\\begin{table}[htbp]\n"
        "\\centering\n"
        + tabular +
        (f"\\caption{{{caption}}}\n")
        + (f"\\label{{{label}}}\n")
        + "\\end{table}"
        )



# Checked: 1
def make_ema_knn_performance_table(
    file_path,
    sheet_name=0,
    metric_columns=None):

    df_results = pd.read_excel(file_path, sheet_name=sheet_name)

    inject_value_list = sorted(df_results["inject_m_EMAs"].unique().tolist())
    n_closest_value_list = sorted(df_results["n_closest_instances"].unique().tolist())

    multiindex_columns = pd.MultiIndex.from_tuples(
        [
            (str_n_neighbours, n_closest_value, metric_name)
            for n_closest_value in n_closest_value_list
            for metric_name in metric_columns
        ]) # Double checked manually.

    df_output = pd.DataFrame(
        index=inject_value_list,
        columns=multiindex_columns,
        dtype=float)

    df_output.index.name = "inject_m_EMAs"
    for _, row_data in df_results.iterrows():
        inject_value = row_data["inject_m_EMAs"]
        n_closest_value = row_data["n_closest_instances"]

        for metric_name in metric_columns:
            df_output.loc[inject_value, (str_n_neighbours, n_closest_value, metric_name)] = row_data[metric_name]


    metric_mapping = {str_precision: 'Prec.',
                      str_recall: str_recall.capitalize(),
                      str_f1_score: 'F1',
                      str_specificity: 'Spec.',
                      str_balanced_acc: 'BA',
                      str_accuracy.lower(): str_accuracy}
    
    df_output = df_output.rename(columns = metric_mapping, level=2)
    df_output.index.name = "N EMAs"

    print(df_to_full_latex_table(df_output, 
                                 caption=os.path.basename(file_path).replace('_', '=='),
                                 label='',
                                 round_upto_n_decimal=2))
        

for file_latex in sorted(os.listdir(loc_restrict_EMA_findings)):
    if 'DS_' in file_latex: continue

    make_ema_knn_performance_table(os.path.join(loc_restrict_EMA_findings, file_latex),
                                metric_columns=[str_precision, str_recall, str_specificity, str_balanced_acc, str_f1_score, str_accuracy.lower()])

In [ ]:
import pandas as pd
import plotly.express as px

# Checked: 1
def plot_balanced_accuracy(file_path, list_y_cols):

    df = pd.read_excel(file_path)
    df.reset_index(inplace = True, drop = True)

    dic_rename = {}
    for col in [str_precision, str_specificity, 'accuracy']:
        dic_rename[col] = col.capitalize()
    
    dic_rename[str_recall] = 'Sensitivity'
    dic_rename[ str_balanced_acc] = 'Balanced Acc.'
    dic_rename[str_f1_score] = 'F1'


    df.rename(columns=dic_rename, inplace= True)
    
    x_col = 'inject_m_EMAs' 
    
    fig = px.line(df, 
                  x=x_col, 
                  y=list_y_cols, 
                  markers=True,
                  title='')
    
    max_ba_row = df.loc[df['Balanced Acc.'].idxmax()]
    best_x = max_ba_row['inject_m_EMAs']

    fig.add_vline(
        x=best_x,
        line_dash="dash",
        line_color="gray",
        opacity=0.7,
        annotation_text="",
        annotation_position="bottom right"
    )

    

    for trace in fig.data:
        if trace.name in ['Sensitivity', 'Specificity']: # 'Balanced Acc.'
            trace.text = [f"{val:.1f}" for val in trace.y]
            trace.mode = 'lines+markers+text' 
            trace.textposition = 'top center'
        else:
            trace.mode = 'lines+markers'


    global_min = df[list_y_cols].min().min()
    global_max = df[list_y_cols].max().max() # .max() get maximum value for each col. .max().max() to get the max value among all cols


    fig.update_layout(yaxis_range=[global_min - 5, global_max + 5])
    fig = update_plotly_layout(fig=fig, x_axis_title="# of EMAs", 
                               y_axis_title="Performance (%)",
                               height=650, width=1200,
                               legend_orientation='h')
    
    fig.update_layout(legend_title_text=None,
                      
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,          
            xanchor="center",
            x=0.4,               
            entrywidth=0.33, # Helps to create two columns in legend
            entrywidthmode="fraction"  #forcing the 6 items to wrap into 2 rows of 3.
        ))
    
    fig.show()


for viz_file in sorted(os.listdir(loc_restrict_EMA_findings)):
    if 'DS' in viz_file: continue

    loc_viz_file = os.path.join(loc_restrict_EMA_findings, viz_file)

    print(viz_file)
    plot_balanced_accuracy(loc_viz_file,
                            list_y_cols=[str_precision.capitalize(), 
                                            'Sensitivity', 
                                            str_specificity.capitalize(), 
                                            'Balanced Acc.', 
                                            'F1', 
                                            'Accuracy'])

# Per P performamce (Revision)

In [ ]:
# Checked: 1
def percent_in_bins(df_in, col_name, start_val=0, end_val=100, step_val=10):

    bin_edges = list( np.arange(start_val, end_val + step_val, step_val) ) # to include np.inf later, I need to convert to list
    bin_labels = [f"{{[{bin_edges[idx]}–{bin_edges[idx+1]})}}" 
                    for idx in range(len(bin_edges)-1)] # checked: range should contain -1

    bin_edges[-1] = np.inf # to include 100
    bin_labels[-1] = bin_labels[-1].replace(')', ']')

    bin_cat = pd.cut(
        df_in[col_name],
        bins=bin_edges,
        labels=bin_labels,
        include_lowest=True,
        right=False  # [0,10), [10,20), ...
    )

    percent = bin_cat.value_counts(normalize=True, dropna=False).mul(100).reindex(bin_labels).fillna(0)
    return percent.rename_axis(str_range).reset_index(name="percent") # Checked manually and played: it is stange to me that I get the values by resetting the index :(


# Checked: 1
def update_plotly_layout(fig, y_axis_title, x_axis_title, legend_orientation = 'h', font_size=21,  legend_x_pos=0.3, legend_y_pos=1.1, height = 400, width=800) -> go.Figure:
    fig.update_layout(plot_bgcolor= "#FFFFFF",
                      yaxis=go.layout.YAxis(title=go.layout.yaxis.Title(text= y_axis_title, font=dict(family='Times New Roman', size= font_size, color='#000000')),
                                            color="#000000", tickfont=dict(family='Times New Roman', size= font_size)),
                      xaxis=go.layout.XAxis(title=go.layout.xaxis.Title(text= x_axis_title, font=dict(family='Times New Roman', size= font_size, color='#000000')), 
                                            color="#000000", tickfont=dict(family='Times New Roman', size= font_size)),
                      legend=dict(orientation = legend_orientation, x=legend_x_pos, y=legend_y_pos,  font=dict(family='Times New Roman', size= font_size), bgcolor="rgba(0,0,0,0)"),
                      
                      font=dict(family="Times New Roman", size= font_size), width = width, height = height)
    return fig




In [ ]:
import plotly.express as px

str_range = 'Range'

# Checked: 1
def check_root_file_perf(df_perf):
    print('\n\nChecking root files performance to make sure that we are using the correct perf file.')
    print(clf_report(true_labels= df_perf[str_actual_class].tolist(),
                     predicted_val_list= df_perf[str_predicted_class].tolist(),
                     bool_print= False))


# Checked: 1
def explore_per_p_performance():
    global df_per_p_perf
    
    df_predict = pd.read_excel("... removed the location a bit for privacy .../Documents/SA39/SAPIENS/State Anxiety/2.5_hours_1_vs_2_10_split/multi_modal/splitted_into3/SAPIENS_HeartRate_Light__accel___magnet_/Performance/ResNet_18/iter_130__a9dfbe4d-4076-48c7-a72b-342fe4c12514__chunk_test_5_val_2__fine_tuned_all_layers/ResNet_18_RQA__fine_tuned_layers__SAPIENS__on_the_same_day__KNeighborsClassifier_incl_predicted_instance_True_global_person_False_add_prev_days_True_including_predicted_class_proba.xlsx")
    check_root_file_perf(df_predict)
    

    df_table = pd.DataFrame(columns=[str_range]) # To be merged with the dfs based on eval metric
    df_per_p_perf = pd.DataFrame()

    for crt_metric in [str_precision, str_recall, str_specificity, str_balanced_acc, str_f1_score, str_accuracy.lower()]:
        list_n_samples = []
        list_df_data = []
        list_metric_based_perf = []
        
        ############ Iterating through each participant and get performance
        print(f'Total # of participants: {len(df_predict[str_p_id].unique())}')
        for pid in sorted(df_predict[str_p_id].unique()):

            mask_pid = df_predict[str_p_id] == pid
            sDf_pre = df_predict[mask_pid].copy() # predict
            
            dic_cls_COUNT = sDf_pre[str_actual_class].astype(str).value_counts().rename({'1': str_recall, '0':str_specificity}).to_dict()

            if (crt_metric == str_recall) and (str_recall not in dic_cls_COUNT.keys()): # manually checked: for all ps there is at least 1 sample of class 1. it is possible to happen if a participant has all classes same and hence, there is no value of calculating the metric when it is not present 
                list_metric_based_perf.append(pd.NA)
                list_n_samples.append(pd.NA)
                continue

            if (crt_metric == str_specificity) and (str_specificity not in dic_cls_COUNT.keys()): ## it is possible to happen if a participant has all classes same.
                list_metric_based_perf.append(pd.NA)
                list_n_samples.append(pd.NA)
                continue

            list_perf, list_cols = clf_report(true_labels = sDf_pre[str_actual_class].tolist(),
                                              predicted_val_list = sDf_pre[str_predicted_class].tolist(), 
                                              bool_print= False)
            
            perf = dict(zip(list_cols, list_perf)).get(crt_metric)
            list_metric_based_perf.append(perf)
            list_df_data.append( perf )
            
            ########## Appending the samples
            if crt_metric in dic_cls_COUNT.keys(): # dic_cls_COUNT.keys() contains only str_recall and str_specificity
                n_samples = dic_cls_COUNT.get(crt_metric)
                list_n_samples.append(n_samples) # Getting the number of instances 

        df_per_p_perf[crt_metric] = list_metric_based_perf
        if crt_metric in [str_recall, str_specificity]:
            df_per_p_perf[crt_metric + 'n_samples'] = list_n_samples

        ########## Bin by range
        bin_perc_df = percent_in_bins(df_in = pd.DataFrame({crt_metric: list_df_data}), 
                                      col_name = crt_metric)
        bin_perc_df.rename(columns={'percent': crt_metric}, inplace = True)
        bin_perc_df[crt_metric] = bin_perc_df[crt_metric].round(2)

        ########## Merging by column
        df_table = pd.merge(bin_perc_df, df_table, on=[str_range], how='left') # left becuase in the first iteration, there will be empty dataframe for the right one.
        df_table.reset_index(drop=True, inplace=True)

        ########## Histogram of samples when the recall/specifcity is < 20
        if crt_metric in [str_recall, str_specificity]:
            hist_samples = px.histogram([val for val in list_n_samples if not pd.isna(val)], title=crt_metric, text_auto=True)
            hist_samples.update_traces(marker = dict(color=list_color[1]), textposition='auto')
            update_plotly_layout(hist_samples, y_axis_title='# of participants', x_axis_title='# of samples')
            hist_samples.update_layout(showlegend = False).show()
    
    df_table.sort_values(by=[str_range], ascending = False, inplace=True)
    print(df_to_full_latex_table(df= df_table, caption='Percentage of participants in each range across evaluation criteria.', label= 'perf_by_p'))

explore_per_p_performance()



In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Checked: 1
def plot_performance_metrics(df):
    
    df = df.drop(columns = [str_specificity+'n_samples', str_recall+'n_samples']).copy()
    
    df.rename(columns = {col: col.capitalize() for col in df.columns}, inplace = True)
    df.rename(columns = {'Bal_acc': 'Balanced Acc.', 'F1_score': 'F1', 'Recall': 'Sensitivity'}, inplace = True)

    fig = go.Figure()

    color = list_color[3]
    violin_fill_color = 'rgba(173, 216, 230, 0.4)'

    for col in df.columns:

        clean_data = df[col].dropna()
        print(col, np.mean(clean_data), np.std(clean_data, ddof=1))
        
        fig.add_trace(go.Violin(
            y= clean_data,
            name=col,
            box_visible=True,    
            meanline_visible=True,  
            points='all',         
            jitter=0.1,            
            pointpos=0,             # Center points over the violin shape
            fillcolor=violin_fill_color,
            line_color=color,    
            marker=dict(
                color="#A6A8AB", 
                opacity=0.6, 
                size=6
            ),
            showlegend=False 
        ))

    fig = update_plotly_layout(fig= fig, y_axis_title='Score', x_axis_title='', height=500, width=1000)
    fig.update_layout(yaxis=dict(
        dtick=20,      
    ))
    fig.show()

    
plot_performance_metrics(df_per_p_perf)

In [ ]:
from scipy import stats

# Checked: 1
def explore_relation(df, metric, samples_col):

    stat, p_metric = stats.shapiro(df[metric].tolist()) # one assumption for pearson is that the data are normally distributed "bivariate normally distributed. In practice the last assumption is checked by requiring both variables to be individually normally distributed (which is a by-product consequence of bivariate normality). " "https://www.statstutor.ac.uk/resources/uploaded/pearsons.pdf"
    stat, p_n_sampels = stats.shapiro(df[samples_col].tolist())

    if (p_metric > 0.05) and (p_n_sampels > 0.05): # Do not have sufficient evidence to reject the null hypothesis. i.e., data maybe normally distributed.
        print('pearsonr')
        print(stats.pearsonr(x= df[metric].tolist(), 
                             y= df[samples_col].tolist()))
    else: # Null hypothesis is rejected. Null hypothesis was: "the data was drawn from a normal distribution."
       print('spearmanr', p_metric, p_n_sampels)
       print(stats.spearmanr(a= df[metric].tolist(), 
                             b = df[samples_col].tolist()))



# Checked: 1
def viz_samples_performance():

    for metric in [str_recall, str_specificity]:
        
        samples_col = metric+'n_samples'

        sDf_perf = df_per_p_perf.copy()
        sDf_perf.dropna(subset=[samples_col], inplace=True)
        sDf_perf.sort_values(by=[samples_col], inplace=True)

        print( samples_col, (sDf_perf[samples_col] < 10).sum() )

        fig = px.scatter(x=sDf_perf[samples_col], y=sDf_perf[metric], title=metric)
        
        if metric == str_recall:
            y_axis_title = 'Sensitivity'
        else:
            y_axis_title = metric.capitalize()
        
        print(metric)
        explore_relation(sDf_perf, metric, samples_col)

        fig = update_plotly_layout(fig= fig, y_axis_title=y_axis_title, x_axis_title='# of samples of a participant', width=1000)
        fig.update_traces(mode="markers")
        fig.show()

viz_samples_performance()

# Operation theater